In [ ]:
import os
import json
import datetime
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from typing import Dict, List, Any, Tuple, Set
from collections import defaultdict
from dotenv import load_dotenv
from datasets import load_dataset
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI
from langchain_xai import ChatXAI
from langchain_core.messages import HumanMessage
from langchain_core.language_models import BaseChatModel
from transformers import AutoTokenizer
import numpy as np
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, retry_if_exception
from module.utils import extract_skeleton

In [ ]:
def load_environment():
    """Load environment variables"""
    load_dotenv()

def load_ground_truth() -> Dict[str, str]:
    """Load ground truth buggy file paths."""
    with open("./ground_truth/bug_paths.json", "r") as f:
        return json.load(f)

def load_combination_results(result_path: str) -> Dict[str, List[str]]:
    """Load combination results file paths."""
    with open(result_path, "r") as f:
        return json.load(f)

def create_llm(llm_name: str, model_name: str, temperature: float, max_tokens: int) -> BaseChatModel:
    """Create language model instance based on configuration"""
    if llm_name == "chatgpt":
        return ChatOpenAI(
            model_name=model_name,
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "claude":
        return ChatAnthropic(
            model_name=model_name,
            anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "deepseek":
        return ChatDeepSeek(
            model=model_name,
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "gemini":
        return ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=os.getenv("GOOGLE_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "mistral":
        return ChatMistralAI(
            model=model_name,
            mistral_api_key=os.getenv("MISTRAL_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_name == "grok":
        return ChatXAI(
            model=model_name,
            api_key=os.getenv("XAI_API_KEY"),
            temperature=temperature,
            max_tokens=max_tokens
        )
    else:
        raise ValueError(f"Unsupported LLM: {llm_name}")

In [ ]:
# def build_verification_prompt(problem_statement: str, file_content: str, file_path: str) -> str:
#     """Build the prompt for verification."""
#     prompt = f"""
# Problem statement:
# {problem_statement}

# File path: {file_path}

# File content:
# {file_content}

# Based on the problem statement and the file content, could fixing this file solve the reported issue?
# Answer with ONLY 'yes' or 'no'.
# """
#     return prompt

def build_verification_prompt(problem_statement: str, file_content: str, file_path: str) -> str:
    """Build the prompt for verification."""
    prompt = f"""
You are an expert software engineer tasked with analyzing potential solutions to reported software issues. Your goal is to determine whether fixing a specific file could solve a given problem.

Here's the problem statement:
<problem_statement>
{problem_statement}
</problem_statement>

The file under consideration is located at:
<file_path>
{file_path}
</file_path>

Here's the content of the file:
<file_content>
{file_content}
</file_content>


Your task is to analyze the problem statement and the file content, then determine if fixing this file could solve the reported issue.

Provide your final answer in the following format::
<answer>yes</answer>

or

<answer>no</answer>

Remember, your final answer must be ONLY 'yes' or 'no' without any additional text.
Do not respond your analysis, onlye return the answer with 'yes' or 'no'.
"""
    return prompt

In [ ]:
# @retry(
#     stop=stop_after_attempt(10),
#     wait=wait_exponential(multiplier=1, min=30, max=300),
#     retry=retry_if_exception_type((Exception))
# )
# def get_llm_response(llm, prompt):
#     """Get response from LLM with retry mechanism"""
#     response = llm.invoke([HumanMessage(content=prompt)])
#     return response.content

def is_429_error(exception):
    return isinstance(exception, Exception) and "429" in str(exception)

@retry(
    stop=stop_after_attempt(10),
    wait=wait_exponential(multiplier=1, min=30, max=180),
    retry=retry_if_exception(is_429_error)
)
def get_llm_response(llm, prompt):
    """Get response from LLM with retry on 429 and fallback to Claude 3.5 Sonnet on other errors."""
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content

    except Exception as e:
        if "429" in str(e):
            # Raise to trigger retry
            print("429")
            print(e)
            raise e

        print(f"Non-retryable error during LLM call ({llm.__class__.__name__} - {getattr(llm, 'model_name', 'unknown')}): {e}")
        print("Switching to Claude 3.5 Sonnet...")

        try:
            fallback_llm = ChatAnthropic(
                model_name="claude-3-5-sonnet-20241022",
                anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
                temperature=llm.temperature,
                max_tokens=llm.max_tokens
            )
            response = fallback_llm.invoke([HumanMessage(content=prompt)])
            return response.content
        except Exception as fallback_error:
            print(f"Fallback Claude 3.5 Sonnet also failed: {fallback_error}")
            raise fallback_error

# @retry(
#     stop=stop_after_attempt(10),
#     wait=wait_exponential(multiplier=1, min=30, max=180),
#     retry=retry_if_exception_type((Exception))
# )
# def get_llm_response(llm, prompt):
#     """Get response from LLM with fallback retry to Claude 3.5 Sonnet on error."""
#     try:
#         response = llm.invoke([HumanMessage(content=prompt)])
#         return response.content

#     except Exception as e:
#         print(f"Error during LLM call ({llm.__class__.__name__} - {getattr(llm, 'model_name', 'unknown')}): {e}")
#         print("Retrying with Claude 3.5 Sonnet...")

#         try:
#             fallback_llm = ChatAnthropic(
#                 model_name="claude-3-5-sonnet-20241022",
#                 anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
#                 temperature=llm.temperature,
#                 max_tokens=llm.max_tokens
#             )
#             response = fallback_llm.invoke([HumanMessage(content=prompt)])
#             return response.content
#         except Exception as fallback_error:
#             print(f"Fallback Claude 3.5 Sonnet also failed: {fallback_error}")
#             raise fallback_error

def process_single_path(task_data: Tuple) -> Dict[str, Any]:
    """Process a single path verification task.
    
    Each task is one LLM verification of a specific path.
    """
    instance_id, predict_path, problem_statement, llm_config, raw_outputs_dir = task_data
    
    llm_name = llm_config["llm"]
    model_name = llm_config["model_name"]
    
    try:
        file_path = os.path.join("./codebases", instance_id, predict_path)
        
        # Check if file exists
        if not os.path.exists(file_path):
            print(f"Warning: File does not exist: {file_path}")
            return {
                "instance_id": instance_id,
                "path": predict_path,
                "llm": llm_name,
                "model_name": model_name,
                "is_yes": False,
                "raw_response": "File does not exist"
            }
        
        # Read file content
        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            file_content = f.read()
        
        # Build the prompt
        prompt = build_verification_prompt(problem_statement, file_content, predict_path)
        
        # Create a new LLM instance for this process
        llm = create_llm(llm_name, model_name, llm_config["temperature"], llm_config["max_tokens"])
        
        # Get LLM response with retry decorator
        raw_response = get_llm_response(llm, prompt)
        
        # Process response
        response_text = raw_response.strip().lower()
        is_yes = "yes" in response_text and "no" not in response_text
        
        if ("yes" in response_text and "no" in response_text) or ("yes" not in response_text and "no" not in response_text):
            print(f"RESPONSE ERROR for {llm_name}/{model_name}: {instance_id}/{predict_path} - Response: {response_text}")
        
        # Save raw response for this specific path
        instance_dir = os.path.join(raw_outputs_dir, instance_id)
        os.makedirs(instance_dir, exist_ok=True)
        
        raw_output_path = os.path.join(instance_dir, f"{predict_path.replace('/', '_')}_{llm_name}_{model_name}.json")
        with open(raw_output_path, "w", encoding="utf-8") as f:
            json.dump({
                "instance_id": instance_id,
                "path": predict_path,
                "llm": llm_name,
                "model_name": model_name,
                "raw_response": raw_response
            }, f, indent=2, ensure_ascii=False)
        
        return {
            "instance_id": instance_id,
            "path": predict_path,
            "llm": llm_name,
            "model_name": model_name,
            "is_yes": is_yes,
            "raw_response": raw_response
        }
        
    except Exception as e:
        print(f"Error verifying {instance_id}/{predict_path} with {llm_name}/{model_name}: {e}")
        return {
            "instance_id": instance_id,
            "path": predict_path,
            "llm": llm_name,
            "model_name": model_name,
            "is_yes": False,
            "raw_response": f"Error: {str(e)}"
        }

In [ ]:
def calculate_initial_accuracy(combination_results: Dict[str, List[str]], ground_truth: Dict[str, str]) -> float:
    """Calculate accuracy of the combination results."""
    correct_count = 0
    total_count = 0
    
    for instance_id, paths in combination_results.items():
        if instance_id not in ground_truth:
            continue
            
        total_count += 1
        gt_path = ground_truth[instance_id]
        
        if gt_path in paths:
            correct_count += 1
    
    return correct_count / total_count if total_count > 0 else 0

def organize_results_by_llm_and_instance(results: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    """Organize results by LLM and instance"""
    organized_results = defaultdict(list)
    
    for result in results:
        llm_name = result["llm"]
        model_name = result["model_name"]
        
        # Create a result entry that matches the original format expected by calculate_aggregate_metrics
        entry = {
            "instance_id": result["instance_id"],
            "predictions": [{
                "path": result["path"],
                "is_yes": result["is_yes"]
            }]
        }
        
        organized_results[(llm_name, model_name)].append(entry)
    
    return organized_results

def transform_to_instance_predictions(results: List[Dict[str, Any]]) -> Dict[str, Dict[str, List[Dict[str, Any]]]]:
    """
    Transform raw results into a nested structure organized by instance_id, llm, and model_name.
    This format is more efficient for later processing.
    """
    # Use a two-level defaultdict where the innermost value is a list
    transformed = defaultdict(lambda: defaultdict(list))
    
    for result in results:
        instance_id = result["instance_id"]
        llm_name = result["llm"]
        model_name = result["model_name"]
        
        # Now this will append to a list, not to a defaultdict
        transformed[instance_id][(llm_name, model_name)].append({
            "path": result["path"],
            "is_yes": result["is_yes"]
        })
    
    return transformed

def calculate_aggregate_metrics(
    transformed_results: Dict[str, Dict[str, List[Dict[str, Any]]]],
    ground_truth: Dict[str, str],
    combination_results: Dict[str, List[str]]
) -> Dict[str, Any]:
    """Calculate metrics for different aggregation methods."""
    # Track paths after aggregation for each instance
    intersect_paths_by_instance = {}
    union_paths_by_instance = {}
    
    # Track metrics
    intersect_correct = 0
    union_correct = 0
    total_count = 0
    
    # Keep track of path counts for average calculation
    intersect_path_counts = []
    union_path_counts = []
    
    # Track zero counts (instances with no paths)
    intersect_zero_count = 0
    union_zero_count = 0
    
    # Get all LLM combinations
    all_llm_keys = set()
    for instance_data in transformed_results.values():
        all_llm_keys.update(instance_data.keys())
    
    # Process each instance
    for instance_id, instance_data in tqdm(transformed_results.items(), desc="Calculating aggregate metrics"):
        if instance_id not in ground_truth:
            continue
            
        total_count += 1
        gt_path = ground_truth[instance_id]
        
        # Get all paths for this instance
        all_paths = set()
        for llm_key, predictions in instance_data.items():
            for pred in predictions:
                all_paths.add(pred["path"])
        
        # Check each path against all LLMs
        intersect_paths = []
        union_paths = []
        
        for path in all_paths:
            # Track if each LLM said yes to this path
            llm_responses = {}
            
            for llm_key in all_llm_keys:
                # Default to no if this LLM didn't verify this path
                llm_responses[llm_key] = False
                
                # Check if this LLM verified this path
                if llm_key in instance_data:
                    # Find the prediction for this path
                    for pred in instance_data[llm_key]:
                        if pred["path"] == path:
                            llm_responses[llm_key] = pred["is_yes"]
                            break
            
            # Check if all LLMs said yes (INTERSECTION)
            if all(llm_responses.values()):
                intersect_paths.append(path)
                
            # Check if at least one LLM said yes (UNION)
            if any(llm_responses.values()):
                union_paths.append(path)
        
        # Store the lists of paths for this instance
        intersect_paths_by_instance[instance_id] = intersect_paths
        union_paths_by_instance[instance_id] = union_paths
        
        # Update path counts for averages
        intersect_path_counts.append(len(intersect_paths))
        union_path_counts.append(len(union_paths))
        
        # Check for empty path sets
        if len(intersect_paths) == 0:
            intersect_zero_count += 1
            
        if len(union_paths) == 0:
            union_zero_count += 1
            
        # Check if ground truth path is in results
        if gt_path in intersect_paths:
            intersect_correct += 1
            
        if gt_path in union_paths:
            union_correct += 1
    
    # Calculate metrics
    intersect_accuracy = intersect_correct / total_count if total_count > 0 else 0
    union_accuracy = union_correct / total_count if total_count > 0 else 0
    
    avg_intersect_paths = np.mean(intersect_path_counts) if intersect_path_counts else 0
    avg_union_paths = np.mean(union_path_counts) if union_path_counts else 0
    
    return {
        "total_instances": total_count,
        "intersect": {
            "correct_instances": intersect_correct,
            "accuracy": intersect_accuracy,
            "avg_remaining_paths": float(avg_intersect_paths),
            "zero_counts": intersect_zero_count,
            "paths_by_instance": intersect_paths_by_instance
        },
        "union": {
            "correct_instances": union_correct,
            "accuracy": union_accuracy,
            "avg_remaining_paths": float(avg_union_paths),
            "zero_counts": union_zero_count,
            "paths_by_instance": union_paths_by_instance
        }
    }

In [ ]:
# Main configuration
config = {
    # Path to load combination results
    "combination_results_path": "./localization_combination_results/union/results/best_top_3_results.json",
    
    # LLM configurations - can define multiple LLMs
    "llms": [
        {
            "llm": "mistral",
            "model_name": "mistral-large-latest",
            "temperature": 0.0,
            "max_tokens": 128
        }
    ],
    
    # Execution configuration
    "num_processes": 1,
}

In [ ]:
# Load environment variables
load_environment()

# Load ground truth
print("Loading ground truth data...")
ground_truth = load_ground_truth()

# Load combination results
print(f"Loading combination results from {config['combination_results_path']}")
combination_results = load_combination_results(config["combination_results_path"])

# Calculate initial accuracy
initial_accuracy = calculate_initial_accuracy(combination_results, ground_truth)
print(f"Initial accuracy of combination results: {initial_accuracy:.4f}")
print(f"Total instances in combination results: {len(combination_results)}")

# Load dataset for problem statements
print("Loading SWE-bench dataset...")
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
dataset_dict = {item["instance_id"]: item for item in dataset}

# Create output directory
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(f"./localization_combination_results/verifications/{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

# Create raw outputs directory
raw_outputs_dir = str(output_dir / "raw_outputs")
os.makedirs(raw_outputs_dir, exist_ok=True)

# Save config and initial results
with open(output_dir / "config.json", "w") as f:
    save_config = config.copy()
    # Add additional information to config
    save_config["timestamp"] = timestamp
    save_config["initial_accuracy"] = initial_accuracy
    save_config["num_instances"] = len(combination_results)
    json.dump(save_config, f, indent=2)

# Prepare tasks - one task per path per LLM
all_tasks = []

for llm_config in config["llms"]:
    llm_name = llm_config["llm"]
    model_name = llm_config["model_name"]
    
    print(f"Preparing tasks for {llm_name}/{model_name}...")
    
    for instance_id, predict_paths in combination_results.items():
        # Skip if instance is not in dataset
        if instance_id not in dataset_dict:
            print(f"Warning: Instance {instance_id} not found in dataset")
            continue
        
        problem_statement = dataset_dict[instance_id]["problem_statement"]
        
        # Create a task for each path (one LLM call per task)
        for predict_path in predict_paths:
            all_tasks.append((
                instance_id, 
                predict_path, 
                problem_statement, 
                llm_config,
                raw_outputs_dir
            ))

# Process tasks with multiprocessing
print(f"Processing {len(all_tasks)} tasks using {config['num_processes']} processes...")

if config["num_processes"] > 1:
    results = []
    with ProcessPoolExecutor(max_workers=config["num_processes"]) as executor:
        # Use tqdm to show progress
        for result in tqdm(executor.map(process_single_path, all_tasks), 
                            total=len(all_tasks), 
                            desc="Processing LLM verifications"):
            results.append(result)
else:
    results = []
    for task in tqdm(all_tasks, desc="Processing LLM verifications"):
        results.append(process_single_path(task))

# Save all raw results
with open(output_dir / "raw_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Transform results for easier processing
print("Organizing results...")
transformed_results = transform_to_instance_predictions(results)

# Save transformed results
with open(output_dir / "transformed_results.json", "w", encoding="utf-8") as f:
    # Convert defaultdict to regular dict for JSON serialization
    serializable_results = {
        instance_id: {
            f"{llm_name}_{model_name}": preds 
            for (llm_name, model_name), preds in llm_data.items()
        }
        for instance_id, llm_data in transformed_results.items()
    }
    json.dump(serializable_results, f, indent=2, ensure_ascii=False)

# Calculate aggregate metrics
print("Calculating aggregated metrics...")
metrics = calculate_aggregate_metrics(transformed_results, ground_truth, combination_results)

# Save metrics
with open(output_dir / "aggregate_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save aggregated paths in the same format as best_top_results.json
print("Saving intersect/union results...")

# Save in the same structure as best_top_results.json
with open(output_dir / "intersect_results.json", "w") as f:
    json.dump(metrics["intersect"]["paths_by_instance"], f, indent=2)
    
with open(output_dir / "union_results.json", "w") as f:
    json.dump(metrics["union"]["paths_by_instance"], f, indent=2)

# Print summary
print("\nResults Summary:")
print(f"Total instances processed: {metrics['total_instances']}")
print("\nInitial combination accuracy: {:.4f}".format(initial_accuracy))
print("\nIntersection method:")
print(f"  Accuracy: {metrics['intersect']['accuracy']:.4f}")
print(f"  Average remaining paths: {metrics['intersect']['avg_remaining_paths']:.4f}")
print(f"  Instances with zero paths: {metrics['intersect']['zero_counts']}")
print("\nUnion method:")
print(f"  Accuracy: {metrics['union']['accuracy']:.4f}")
print(f"  Average remaining paths: {metrics['union']['avg_remaining_paths']:.4f}")
print(f"  Instances with zero paths: {metrics['union']['zero_counts']}")
print(f"\nResults saved to {output_dir}")